# SMILEPhysicsList: 物理モデル

Geant4 で使う物理過程(電磁相互作用、ハドロン、中性子、崩壊、Gd の中性子捕獲)を決めます。設定ファイルの `physics` の節です。

| | |
| --- | --- |
| SMILE_tool_box | `SMILE_PhysicsList`(sim.mac の `GdPhysics`) |
| ソース | `simulation_source/modules/SMILEPhysicsList/`(Gd の捕獲のモデルは `simulation_source/vendor/gd_capture/`) |
| 基底クラス | anlgeant4 の `VANLPhysicsList` |

## 1. 何をするか

次の物理過程を登録します(`SMILEPhysicsList.cc`)。

| 物理過程 | 中身 |
| --- | --- |
| `G4EmLivermorePhysics`(`em_msc = gs`)/ `SMILEEmLivermoreUrbanPhysics`(`urban`) | 低いエネルギーまで正確な電磁相互作用(光電吸収、コンプトン散乱のドップラー広がりなど)。e± の多重散乱だけが違う |
| `G4DecayPhysics`, `G4RadioactiveDecayPhysics` | 崩壊、放射性崩壊 |
| `G4HadronElasticPhysicsHP`, `G4HadronPhysicsFTFP_BERT_HP` | ハドロン。20 MeV 以下の中性子は評価済みの断面積のデータ(G4NDL、`environment.G4NEUTRONHPDATA`)を使う |
| `GdNeutronCapturePhysics`(`gd_physics = true`) | Gd の中性子捕獲のガンマを ANNRI-Gd 系のモデル(離散のカスケード + 連続成分の表 `gd155_table`, `gd157_table`)で作る |
| `G4StoppingPhysics`, `G4IonPhysics`, `G4StepLimiterPhysics` | 止まった粒子の吸収、イオン、ステップの長さの上限(`StepLimit` モジュール用) |

二次粒子を作るかどうかのカットは飛程 `cut_value`(1 µm)で、カットの表は `low_energy_limit`(250 eV)からです。
物理モデルの選び方とその影響は `config/README.md` の「物理モデルの選択肢」と `docs/simulation_port_log.md` にあります。

## 2. パラメータ

| パラメータ | 型 | 既定値(C++) | この設定の値 | 単位 | 説明 |
| --- | --- | --- | --- | --- | --- |
| `em_msc` | string | gs | gs |  | e± の多重散乱: gs(Geant4 11 の G4EmLivermorePhysics)/ urban(SMILE_tool_box = Geant4 10.4 と同じ) |
| `gd_physics` | bool | false | true |  | true: Gd の中性子捕獲のガンマを ANNRI-Gd 系のモデルで作る(sim.mac の GdPhysics 1) |
| `gd155_table` | string |  | {sim_data}/Gd_capture/Gd155.dat |  | Gd の捕獲のモデルの連続成分の表(Gd-155) |
| `gd157_table` | string |  | {sim_data}/Gd_capture/Gd157_org.dat |  | Gd の捕獲のモデルの連続成分の表(Gd-157) |
| `cut_value` | double | 0.001 | 0.001 | mm | γ / e- / e+ の生成のカット(飛程)[mm] |
| `low_energy_limit` | double | 0.25 | 0.25 | keV | カットの表のエネルギーの下限 [keV] |
| `verbose` | int | 0 | (設定に無い: 既定値) |  | Geant4 の物理リストの表示の詳しさ |

中性子のデータの場所は `physics` ではなく `environment.G4NEUTRONHPDATA`(環境変数)で選びます。Geant4 が物理の表を作る前に設定する必要があるためです。

パラメータは設定ファイル `config/simulation/smile2p_flight_parma_thermal_neutron_d10_raw.json` の `physics.parameters` に書きます
(書き方は `config/README.md`)。1 回だけ変えて試すときは `run_sim.py ... --set physics.em_msc=urban` で上書きします。
「既定値(C++)」は設定ファイルに書かなかったときの値です。

## 3. ほかのモジュールとのやり取り

ほかのモジュールに値を渡す関数はありません。登録した物理過程は Geant4 の中で使われます。
`EdepPickUp` は、TPC のステップごとに、その粒子を作った物理過程の名前(`tpc_creator`)を記録します。

## 4. 確かめる

TPC にエネルギーを落とした粒子が、どの物理過程で作られたかを数えます(熱中性子の例)。

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ROOT
ROOT.gROOT.SetBatch(True)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_sim.py").exists())
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CONFIG = "smile2p_flight_parma_thermal_neutron_d10_raw"
cfg = ec.load_sim(CONFIG)
# 1_DataIO/DataIO_example.ipynb の出力(d10 の熱中性子、100 万個、seed 1)を使う
SIM = REPO / "tutorials_simulation" / "cache" / "d10_seed1_1M"
EDEP, RAW = SIM / "edep.root", SIM / "raw"
assert EDEP.exists(), "先に ../../DataIO_example.ipynb を実行してください"

def flat(column):
    # RDataFrame.AsNumpy() の vector 型の列 -> 全イベントをつないだ 1 次元の numpy 配列
    return np.concatenate([np.asarray(v) for v in column]) if len(column) else np.array([])

In [2]:
d = ROOT.RDataFrame("edep", str(EDEP)).Filter("e_tpc > 0").AsNumpy(["tpc_creator", "tpc_pdg", "tpc_e"])
creator = np.concatenate([np.array([str(s) for s in v]) for v in d["tpc_creator"]])
pdg, e = flat(d["tpc_pdg"]), flat(d["tpc_e"])
print("TPC にエネルギーを落としたイベント:", len(d["tpc_e"]))
print(f"{'作った物理過程':22s} {'ステップ':>9s} {'エネルギー [keV]':>16s}")
for name in sorted(set(creator), key=lambda n: -e[creator == n].sum()):
    m = creator == name
    print(f"{name:22s} {m.sum():9d} {e[m].sum():16.1f}")
print("粒子(PDG):", {int(k): int((pdg == k).sum()) for k in np.unique(pdg)})

TPC にエネルギーを落としたイベント: 310
作った物理過程                     ステップ      エネルギー [keV]
compt                     284195          13894.8
eIoni                      57564          12061.2
phot                       19006           1217.1
conv                       27080           1213.1
RadioactiveDecay            3015            158.4
primary                       69             32.2
nCapture                      41              1.4
eBrem                          6              0.4
annihil                        7              0.2
粒子(PDG): {-11: 18437, 11: 372409, 22: 65, 2112: 69, 1000180410: 1, 1000190410: 1, 1000190411: 1}


- `compt`(コンプトン散乱)、`phot`(光電吸収)、`conv`(電子対生成)、`eIoni`(電離で叩き出された電子)などは電磁相互作用(`G4EmLivermorePhysics`)、
  `nCapture` は中性子の捕獲(Gd の捕獲は `gd_physics`)です。
- 熱中性子のシミュレーションで TPC にエネルギーを落とすのは、ほとんどが電子(PDG 11)です。中性子が捕獲されて出たガンマ線が、
  ガスや周りの物質で電子を叩き出します。